# Tahoe SLM Training (Colab, QLoRA)

This notebook fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` on synthetic extraction data and pushes a LoRA adapter to Hugging Face.

In [ ]:
!pip -q install --upgrade pip
!pip -q install unsloth datasets transformers trl peft accelerate bitsandbytes huggingface_hub

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get('HF_TOKEN', '')  # or paste token here
if not HF_TOKEN:
    raise ValueError('Set HF_TOKEN env var in Colab before running')
login(token=HF_TOKEN)
print('HF login ok')

In [ ]:
from datasets import load_dataset

TRAIN_URL = 'https://huggingface.co/datasets/hariharan5693/tahoe-synthetic-extraction-v1/resolve/main/train_chat.jsonl'
VALID_URL = 'https://huggingface.co/datasets/hariharan5693/tahoe-synthetic-extraction-v1/resolve/main/valid_chat.jsonl'

train_ds = load_dataset('json', data_files=TRAIN_URL, split='train')
valid_ds = load_dataset('json', data_files=VALID_URL, split='train')

print('train', len(train_ds), 'valid', len(valid_ds))
print(train_ds[0].keys())

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
)
print('model ready')

In [ ]:
def to_text(example):
    msgs = example['messages']
    txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': txt}

train_text = train_ds.map(to_text, remove_columns=train_ds.column_names)
valid_text = valid_ds.map(to_text, remove_columns=valid_ds.column_names)

print(train_text[0]['text'][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=valid_text,
    dataset_text_field='text',
    max_seq_length=MAX_LEN,
    args=TrainingArguments(
        output_dir='tahoe_qwen15b_lora_out',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=20,
        evaluation_strategy='steps',
        eval_steps=100,
        save_steps=100,
        fp16=True,
        optim='adamw_torch',
        lr_scheduler_type='linear',
        report_to='none',
    ),
)

trainer.train()

In [ ]:
MODEL_REPO = 'hariharan5693/tahoe-qwen25-1p5b-extractor-lora-v1'

model.push_to_hub(MODEL_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(MODEL_REPO, token=HF_TOKEN)
print('Pushed model to:', f'https://huggingface.co/{MODEL_REPO}')